# Experiment 7 — Semantic-Only Retrieval

**Approach:** Use a pretrained multilingual sentence embedding model (`paraphrase-multilingual-MiniLM-L12-v2`) to encode all training questions into dense vectors. For each test question, retrieve the most semantically similar training question and return its answer.

**No fine-tuning required** — this experiment runs entirely on CPU/GPU encoding and cosine similarity search.

**Expected runtime:** 20–40 minutes on Colab.

## 1 — Install and Import Packages

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# Install required packages
!pip install -q scikit-learn pandas numpy rouge-score
!pip install -q sentence-transformers

print('✅ Packages installed')

  Preparing metadata (setup.py) ... done
✅ Packages installed


In [2]:
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

warnings.filterwarnings('ignore')

SEED = 42
print('✅ Imports done')

✅ Imports done


## 2 — Set File Paths

In [3]:
DATA_DIR = Path('/content')

TRAIN_PATH      = DATA_DIR / 'Train.csv'
TEST_PATH       = DATA_DIR / 'Test.csv'
VAL_PATH        = DATA_DIR / 'Val.csv'
SAMPLE_SUB_PATH = DATA_DIR / 'SampleSubmission.csv'

OUTPUT_EXP7 = Path('/content/submission_exp7_semantic.csv')

for path in [TRAIN_PATH, TEST_PATH, VAL_PATH, SAMPLE_SUB_PATH]:
    status = '✅' if path.exists() else '❌'
    print(f'{status} {path}')

✅ /content/Train.csv
✅ /content/Test.csv
✅ /content/Val.csv
✅ /content/SampleSubmission.csv


## 3 — Load Data

In [4]:
train             = pd.read_csv(TRAIN_PATH)
test              = pd.read_csv(TEST_PATH)
val               = pd.read_csv(VAL_PATH)
sample_submission = pd.read_csv(SAMPLE_SUB_PATH)

print(f'Train shape : {train.shape}')
print(f'Test shape  : {test.shape}')
print(f'Val shape   : {val.shape}')

print('\nLanguage distribution in training set:')
display(train['subset'].value_counts())

Train shape : (29815, 4)
Test shape  : (2618, 3)
Val shape   : (6686, 4)

Language distribution in training set:


,count
subset,
Eng_Uga,7624
Aka_Gha,4455
Eng_Gha,4443
Eng_Eth,3915
Lug_Uga,3383
Eng_Ken,2080
Swa_Ken,2070
Amh_Eth,1845


## 4 — Column Configuration

In [5]:
ID_COL            = 'ID'
TEST_ID_COL       = 'ID'
QUESTION_COL      = 'input'
TEST_QUESTION_COL = 'input'
ANSWER_COL        = 'output'
LANG_COL          = 'subset'
TEST_LANG_COL     = 'subset'

# Semantic model
SEMANTIC_MODEL_NAME = 'paraphrase-multilingual-MiniLM-L12-v2'
SEMANTIC_BATCH_SIZE = 256

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {DEVICE}')
print(f'Model  : {SEMANTIC_MODEL_NAME}')

Device : cpu
Model  : paraphrase-multilingual-MiniLM-L12-v2


## 5 — Text Cleaning

In [6]:
def clean_text(x):
    if pd.isna(x):
        return ''
    return str(x).strip()

train[QUESTION_COL]      = train[QUESTION_COL].map(clean_text)
train[ANSWER_COL]        = train[ANSWER_COL].map(clean_text)
test[TEST_QUESTION_COL]  = test[TEST_QUESTION_COL].map(clean_text)
val[QUESTION_COL]        = val[QUESTION_COL].map(clean_text)
val[ANSWER_COL]          = val[ANSWER_COL].map(clean_text)

print('✅ Text cleaned')

✅ Text cleaned


## 6 — ROUGE Evaluation

In [7]:
try:
    from rouge_score import rouge_scorer

    class WhitespaceTokenizer:
        def tokenize(self, text):
            if text is None:
                return []
            return str(text).strip().split()

    _scorer = rouge_scorer.RougeScorer(
        ['rouge1', 'rougeL'],
        tokenizer=WhitespaceTokenizer()
    )

    def compute_rouge(predictions, references):
        r1_scores, rl_scores = [], []
        for pred, ref in zip(predictions, references):
            scores = _scorer.score(str(ref), str(pred))
            r1_scores.append(scores['rouge1'].fmeasure)
            rl_scores.append(scores['rougeL'].fmeasure)
        return {
            'rouge1_f1': float(np.mean(r1_scores)),
            'rougeL_f1': float(np.mean(rl_scores)),
        }

    def compute_rouge_by_language(predictions, references, languages):
        rows = []
        for lang in sorted(set(languages)):
            idx = [i for i, l in enumerate(languages) if l == lang]
            preds_l = [predictions[i] for i in idx]
            refs_l  = [references[i]  for i in idx]
            m = compute_rouge(preds_l, refs_l)
            rows.append({'language': lang, 'rouge1_f1': m['rouge1_f1'], 'rougeL_f1': m['rougeL_f1']})
        return pd.DataFrame(rows).set_index('language')

    print('✅ ROUGE scoring available')
    compute_rouge_available = True

except Exception as e:
    print(f'⚠️  ROUGE unavailable: {e}')
    compute_rouge_available = False
    compute_rouge = None

✅ ROUGE scoring available


## 7 — Semantic Retrieval Index

In [8]:
class SemanticRetrievalIndex:
    """
    Semantic-only retrieval using sentence embeddings.
    Encodes all training questions, then for each test question
    finds the most similar training question via cosine similarity
    and returns its answer.
    """

    def __init__(self, model_name=SEMANTIC_MODEL_NAME, batch_size=SEMANTIC_BATCH_SIZE):
        self.model_name = model_name
        self.batch_size = batch_size
        self.encoder    = None
        self.questions  = []
        self.answers    = []
        self.embeddings = None
        self.subset_indices = {}

    def _get_encoder(self):
        if self.encoder is None:
            print(f'  Loading encoder: {self.model_name} ({DEVICE})')
            self.encoder = SentenceTransformer(self.model_name, device=DEVICE)
        return self.encoder

    def _encode(self, texts):
        encoder = self._get_encoder()
        parts = []
        for start in range(0, len(texts), self.batch_size):
            end = min(start + self.batch_size, len(texts))
            batch = texts[start:end]
            parts.append(
                encoder.encode(batch, show_progress_bar=False, normalize_embeddings=True)
            )
            if end % 5000 == 0 or end == len(texts):
                print(f'    Encoded {end:,} / {len(texts):,}')
        return np.vstack(parts)

    def fit(self, df, question_col, answer_col, lang_col):
        self.questions = df[question_col].fillna('').astype(str).tolist()
        self.answers   = df[answer_col].fillna('').astype(str).tolist()

        print(f'  Encoding {len(self.questions):,} training questions...')
        self.embeddings = self._encode(self.questions)

        # Build per-subset index for faster retrieval
        for subset in df[lang_col].unique():
            mask = df[lang_col] == subset
            self.subset_indices[subset] = np.where(mask.values)[0]

        print(f'  ✅ Index built: {len(self.questions):,} questions, {len(self.subset_indices)} subsets')
        return self

    def retrieve_one(self, question_embedding, subset):
        if subset in self.subset_indices:
            indices = self.subset_indices[subset]
        else:
            indices = np.arange(len(self.questions))

        candidate_embeddings = self.embeddings[indices]
        scores = cosine_similarity(question_embedding.reshape(1, -1), candidate_embeddings).flatten()
        best = int(indices[int(np.argmax(scores))])
        return self.answers[best]

    def predict(self, df, question_col, lang_col):
        questions = df[question_col].fillna('').astype(str).tolist()
        subsets   = df[lang_col].tolist()

        print(f'  Encoding {len(questions):,} query questions...')
        query_embeddings = self._encode(questions)

        predictions = []
        for i, (emb, subset) in enumerate(zip(query_embeddings, subsets)):
            predictions.append(self.retrieve_one(emb, subset))
            if (i + 1) % 500 == 0 or (i + 1) == len(questions):
                print(f'  Retrieved {i + 1:,} / {len(questions):,}')

        return predictions

print('✅ SemanticRetrievalIndex defined')

✅ SemanticRetrievalIndex defined


## 8 — Validate on Val Set

In [9]:
print('Building semantic index on training data...')
index_val = SemanticRetrievalIndex().fit(train, QUESTION_COL, ANSWER_COL, LANG_COL)

# Sample 500 for speed
val_sample = val.sample(n=500, random_state=SEED)
print(f'\nPredicting on {len(val_sample)} validation examples...')
val_preds = index_val.predict(val_sample, QUESTION_COL, LANG_COL)
val_refs  = val_sample[ANSWER_COL].tolist()

if compute_rouge_available:
    metrics = compute_rouge(val_preds, val_refs)
    print(f'\n📊 Exp 7 — Semantic Retrieval — Validation ROUGE')
    print(f'   ROUGE-1 F1 : {metrics["rouge1_f1"]:.4f}')
    print(f'   ROUGE-L F1 : {metrics["rougeL_f1"]:.4f}')

    print('\n📊 ROUGE by language:')
    lang_metrics = compute_rouge_by_language(val_preds, val_refs, val_sample[LANG_COL].tolist())
    display(lang_metrics.round(4))

Building semantic index on training data...
  Encoding 29,815 training questions...
  Loading encoder: paraphrase-multilingual-MiniLM-L12-v2 (cpu)


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

    Encoded 29,815 / 29,815
  ✅ Index built: 29,815 questions, 8 subsets

Predicting on 500 validation examples...
  Encoding 500 query questions...
    Encoded 500 / 500
  Retrieved 500 / 500

📊 Exp 7 — Semantic Retrieval — Validation ROUGE
   ROUGE-1 F1 : 0.3846
   ROUGE-L F1 : 0.3340

📊 ROUGE by language:


,rouge1_f1,rougeL_f1
language,,
Aka_Gha,0.2588,0.1600
Amh_Eth,0.0854,0.0770
Eng_Eth,0.5403,0.5215
Eng_Gha,0.2440,0.1590
Eng_Ken,0.7185,0.6889
Eng_Uga,0.6330,0.5958
Lug_Uga,0.2336,0.1995
Swa_Ken,0.4485,0.4070


## 9 — Generate Test Predictions

In [10]:
# Fit on train + val combined for best coverage
corpus = pd.concat([train, val], ignore_index=True)
print(f'Building final index on {len(corpus):,} examples (train + val)...')
index_test = SemanticRetrievalIndex().fit(corpus, QUESTION_COL, ANSWER_COL, LANG_COL)

print(f'\nPredicting on {len(test):,} test questions...')
test_preds = index_test.predict(test, TEST_QUESTION_COL, TEST_LANG_COL)

print(f'\n✅ Generated {len(test_preds):,} answers')

# Preview
preview = test[[TEST_ID_COL, TEST_QUESTION_COL]].copy()
preview['predicted_answer'] = test_preds
display(preview.head(5))

Building final index on 36,501 examples (train + val)...
  Encoding 36,501 training questions...
  Loading encoder: paraphrase-multilingual-MiniLM-L12-v2 (cpu)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

    Encoded 36,501 / 36,501
  ✅ Index built: 36,501 questions, 8 subsets

Predicting on 2,618 test questions...
  Encoding 2,618 query questions...
    Encoded 2,618 / 2,618
  Retrieved 500 / 2,618
  Retrieved 1,000 / 2,618
  Retrieved 1,500 / 2,618
  Retrieved 2,000 / 2,618
  Retrieved 2,500 / 2,618
  Retrieved 2,618 / 2,618

✅ Generated 2,618 answers


,ID,input,predicted_answer
0,ID_TS_Aka_Gha_A3B1799D,"Fa nneɛma a wɔde bɛyɛ nkyerɛkyerɛ nneɛma, adwu...",wo sukuu ɔyarehwɛfo anaa apomoden ho dwumayɛni...
1,ID_TS_Aka_Gha_1C80317F,Dɛn ne nea ebetumi afi hokwan a mmabun wɔ sɛ w...,Ahintasɛm a wobu so betumi de ɔhaw kɛse aba mm...
2,ID_TS_Aka_Gha_06671AD1,Akwan bɛn na mmabun bɛtumi afa so ehunu nsusua...,Kamfo mmabun wɔ su pa ne nkɔso a wɔanya mu (Pr...
3,ID_TS_Aka_Gha_BDD640FB,"Sɛnea amammerɛ mu mmra, asetena mu suban, ne t...",Nhomasua pa a wobenya: Nhomasua ho nneɛma ne h...
4,ID_TS_Aka_Gha_46685257,Adɛn nti na mmara nsesaeɛ 'policy advocacy' ho...,Sika mmoa (Funding/grants). Afotu ne nteteɛ (M...


## 10 — Save Submission

In [11]:
def make_submission(ids, predictions, output_path):
    cleaned = [
        re.sub(r'\s+', ' ', str(p)).strip() if p else 'no answer'
        for p in predictions
    ]
    sub = pd.DataFrame({
        'ID'        : ids,
        'TargetRLF1': cleaned,
        'TargetR1F1': cleaned,
        'TargetLLM' : cleaned,
    })
    sub.to_csv(output_path, index=False)
    print(f'✅ Submission saved to: {output_path}')
    print(f'   Shape : {sub.shape}')
    display(sub.head())
    return sub

make_submission(
    ids         = test[TEST_ID_COL].values,
    predictions = test_preds,
    output_path = OUTPUT_EXP7,
)

✅ Submission saved to: /content/submission_exp7_semantic.csv
   Shape : (2618, 4)


,ID,TargetRLF1,TargetR1F1,TargetLLM
0,ID_TS_Aka_Gha_A3B1799D,wo sukuu ɔyarehwɛfo anaa apomoden ho dwumayɛni...,wo sukuu ɔyarehwɛfo anaa apomoden ho dwumayɛni...,wo sukuu ɔyarehwɛfo anaa apomoden ho dwumayɛni...
1,ID_TS_Aka_Gha_1C80317F,Ahintasɛm a wobu so betumi de ɔhaw kɛse aba mm...,Ahintasɛm a wobu so betumi de ɔhaw kɛse aba mm...,Ahintasɛm a wobu so betumi de ɔhaw kɛse aba mm...
2,ID_TS_Aka_Gha_06671AD1,Kamfo mmabun wɔ su pa ne nkɔso a wɔanya mu (Pr...,Kamfo mmabun wɔ su pa ne nkɔso a wɔanya mu (Pr...,Kamfo mmabun wɔ su pa ne nkɔso a wɔanya mu (Pr...
3,ID_TS_Aka_Gha_BDD640FB,Nhomasua pa a wobenya: Nhomasua ho nneɛma ne h...,Nhomasua pa a wobenya: Nhomasua ho nneɛma ne h...,Nhomasua pa a wobenya: Nhomasua ho nneɛma ne h...
4,ID_TS_Aka_Gha_46685257,Sika mmoa (Funding/grants). Afotu ne nteteɛ (M...,Sika mmoa (Funding/grants). Afotu ne nteteɛ (M...,Sika mmoa (Funding/grants). Afotu ne nteteɛ (M...


,ID,TargetRLF1,TargetR1F1,TargetLLM
0,ID_TS_Aka_Gha_A3B1799D,wo sukuu ɔyarehwɛfo anaa apomoden ho dwumayɛni...,wo sukuu ɔyarehwɛfo anaa apomoden ho dwumayɛni...,wo sukuu ɔyarehwɛfo anaa apomoden ho dwumayɛni...
1,ID_TS_Aka_Gha_1C80317F,Ahintasɛm a wobu so betumi de ɔhaw kɛse aba mm...,Ahintasɛm a wobu so betumi de ɔhaw kɛse aba mm...,Ahintasɛm a wobu so betumi de ɔhaw kɛse aba mm...
2,ID_TS_Aka_Gha_06671AD1,Kamfo mmabun wɔ su pa ne nkɔso a wɔanya mu (Pr...,Kamfo mmabun wɔ su pa ne nkɔso a wɔanya mu (Pr...,Kamfo mmabun wɔ su pa ne nkɔso a wɔanya mu (Pr...
3,ID_TS_Aka_Gha_BDD640FB,Nhomasua pa a wobenya: Nhomasua ho nneɛma ne h...,Nhomasua pa a wobenya: Nhomasua ho nneɛma ne h...,Nhomasua pa a wobenya: Nhomasua ho nneɛma ne h...
4,ID_TS_Aka_Gha_46685257,Sika mmoa (Funding/grants). Afotu ne nteteɛ (M...,Sika mmoa (Funding/grants). Afotu ne nteteɛ (M...,Sika mmoa (Funding/grants). Afotu ne nteteɛ (M...
...,...,...,...,...
2613,ID_TS_Swa_Ken_C9525185,Chunusi zinazowasha kwenye uume zinaweza kuwa ...,Chunusi zinazowasha kwenye uume zinaweza kuwa ...,Chunusi zinazowasha kwenye uume zinaweza kuwa ...
2614,ID_TS_Swa_Ken_FD034ADA,"""Maambukizi ya virusi vya ukimwi (Virusi vya U...","""Maambukizi ya virusi vya ukimwi (Virusi vya U...","""Maambukizi ya virusi vya ukimwi (Virusi vya U..."
2615,ID_TS_Swa_Ken_81F38DD4,Ikiwa mtu unayemjua anapambana na woga au hisi...,Ikiwa mtu unayemjua anapambana na woga au hisi...,Ikiwa mtu unayemjua anapambana na woga au hisi...
2616,ID_TS_Swa_Ken_8DDCE719,"Kulingana na Shirika la Afya Duniani (WHO), wa...","Kulingana na Shirika la Afya Duniani (WHO), wa...","Kulingana na Shirika la Afya Duniani (WHO), wa..."
